In [1]:
import os, json, random
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, AllChem, DataStructs
import selfies
from tqdm import tqdm
from rdkit.Chem import PandasTools
from itertools import product

In [2]:
device = torch.device("cpu")
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

Using device: cpu


In [3]:
# --------------------
# lr Parse Setup
# --------------------
parser = argparse.ArgumentParser()
parser.add_argument('--lr_g', type=float, required=True)
parser.add_argument('--lr_d', type=float, required=True)
args = parser.parse_args()

lr_g = args.lr_g
lr_d = args.lr_d
# --------------------
# Output Directory Setup
# --------------------
output_dir = f"outputs/lrg_{lr_g}_lrd_{lr_d}"
os.makedirs(output_dir, exist_ok=True)
checkpoint_path = os.path.join(output_dir, "checkpoint_batch.pth")
metrics_path = os.path.join(output_dir, "training_metrics_batch.csv")

usage: ipykernel_launcher.py [-h] --lr_g LR_G --lr_d LR_D
ipykernel_launcher.py: error: the following arguments are required: --lr_g, --lr_d


SystemExit: 2

/opt/conda/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3386: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [4]:
# Load QM9 CSV file (make sure qm9.csv is in the same folder)
df_acid = pd.read_csv('lewis_acid_candidates.csv')
df_base = pd.read_csv('zinc.csv')

# Ensure it has a column named 'smiles'
assert 'smiles' in df_acid.columns, "CSV must have a 'smiles' column."
assert 'smiles' in df_base.columns, "CSV must have a 'smiles' column."

# Convert SMILES to RDKit Mol objects
PandasTools.AddMoleculeColumnToFrame(df_acid, smilesCol='smiles')
PandasTools.AddMoleculeColumnToFrame(df_base, smilesCol='smiles')

def generate_flp_pairs(df_acid, df_base, acid_col='smiles', base_col='smiles'):
    """
    Generate all unique Lewis acid–base pairs as FLP SMILES strings.

    Args:
        df_acid (pd.DataFrame): DataFrame with Lewis acids (must include 'smiles' column).
        df_base (pd.DataFrame): DataFrame with Lewis bases (must include 'smiles' column).
        acid_col (str): Column name for acid SMILES.
        base_col (str): Column name for base SMILES.

    Returns:
        pd.DataFrame: DataFrame with 'flp_smiles' and 'type' columns.
    """
    assert acid_col in df_acid.columns, f"Acid DataFrame must contain column '{acid_col}'"
    assert base_col in df_base.columns, f"Base DataFrame must contain column '{base_col}'"

    acids = df_acid[acid_col].unique()
    bases = df_base[base_col].unique()

    print(f"🔬 Generating all pairwise combinations: {len(acids)} acids × {len(bases)} bases")

    flp_data = [{'flp_smiles': f"{b}.{a}", 'type': 'inter'} for a, b in product(acids, bases)]
    flp_df = pd.DataFrame(flp_data)

    print(f"✅ Generated {len(flp_df)} FLP pairs.")

    return flp_df

# Example use:
print("Extracting all FLP candidates (intra + inter)...")
flp_all_df = generate_flp_pairs(df_acid, df_base)
print(flp_all_df.head())

Extracting all FLP candidates (intra + inter)...
🔬 Generating all pairwise combinations: 165 acids × 51300 bases
✅ Generated 8464500 FLP pairs.
                                          flp_smiles   type
0  Fc1ccc(Cn2c(N3CCNCC3)nc3ccccc32)cc1.[B](c1ccc(...  inter
1  Cc1ccc(CC2(O)CCN(CCOc3ccc(O)cc3)CC2)cc1.[B](c1...  inter
2  Clc1cccc2c1CN(C1=NCCN1)C2.[B](c1ccc(F)cc1)(c1c...  inter
3  CC1(C)CO[C@@H](CC(=O)O)CN1.[B](c1ccc(F)cc1)(c1...  inter
4  Cc1noc(NS(=O)(=O)c2cccc3c(N(C)C)cccc23)c1C.[B]...  inter


In [5]:
def extract_flp_features(flp_smiles):
    features = {
        'charge_diff': None,
        'tpsa': None,
        'rotatable_bonds': None,
        'num_rings': None
    }

    try:
        if '.' in flp_smiles:
            # Intermolecular FLP pair: split into base + acid
            base_smiles, acid_smiles = flp_smiles.split('.')
            base_mol = Chem.MolFromSmiles(base_smiles)
            acid_mol = Chem.MolFromSmiles(acid_smiles)
            if base_mol is None or acid_mol is None:
                return features

            # Add Hs + charges
            base = Chem.AddHs(base_mol)
            acid = Chem.AddHs(acid_mol)
            ComputeGasteigerCharges(base)
            ComputeGasteigerCharges(acid)

            base_charges = [float(atom.GetProp('_GasteigerCharge')) for atom in base.GetAtoms()]
            acid_charges = [float(atom.GetProp('_GasteigerCharge')) for atom in acid.GetAtoms()]
            features['charge_diff'] = abs(max(base_charges) - min(acid_charges))

            # Sum properties
            features['tpsa'] = Descriptors.TPSA(base) + Descriptors.TPSA(acid)
            features['rotatable_bonds'] = Descriptors.NumRotatableBonds(base) + Descriptors.NumRotatableBonds(acid)
            features['num_rings'] = base.GetRingInfo().NumRings() + acid.GetRingInfo().NumRings()

        else:
            # Intramolecular FLP: single molecule
            mol = Chem.MolFromSmiles(flp_smiles)
            if mol is None:
                return features

            mol = Chem.AddHs(mol)
            ComputeGasteigerCharges(mol)
            charges = [float(atom.GetProp('_GasteigerCharge')) for atom in mol.GetAtoms()]
            # Charge diff: max minus min in same molecule
            features['charge_diff'] = abs(max(charges) - min(charges))

            # Single-molecule descriptors
            features['tpsa'] = Descriptors.TPSA(mol)
            features['rotatable_bonds'] = Descriptors.NumRotatableBonds(mol)
            features['num_rings'] = mol.GetRingInfo().NumRings()

    except Exception as e:
        print("⚠️ Feature extraction failed for:", flp_smiles, "| Error:", e)

    return features

In [6]:
class MolecularSELFIESDataset(Dataset):
    def __init__(self, smiles_list):
        self.smiles = smiles_list

        # Convert each SMILES to its tokenized SELFIES representation
        self.selfies_tokens_list = [self.smiles_to_selfies_tokens(smi) for smi in self.smiles]

        # Build vocabulary from all tokens (reserve index 0 for padding)
        all_tokens = [token for tokens in self.selfies_tokens_list for token in tokens]
        unique_tokens = sorted(set(all_tokens))
        self.token_to_idx = {token: idx + 1 for idx, token in enumerate(unique_tokens)}
        self.idx_to_token = {idx: token for token, idx in self.token_to_idx.items()}

        # Determine maximum sequence length for padding
        self.max_seq_len = max(len(tokens) for tokens in self.selfies_tokens_list)
        self.vocab_size = len(self.token_to_idx) + 1  # +1 for padding

        # Convert token sequences to fixed-length one-hot encoded vectors (flattened)
        self.encoded_data = []
        for tokens in self.selfies_tokens_list:
            token_indices = [self.token_to_idx[token] for token in tokens]
            padded = token_indices + [0] * (self.max_seq_len - len(token_indices))
            one_hot = np.eye(self.vocab_size)[padded]
            one_hot_flat = one_hot.flatten()
            self.encoded_data.append(one_hot_flat)

    def smiles_to_selfies_tokens(self, smiles):
        """Convert SMILES to SELFIES tokens"""
        try:
            selfies_str = selfies.encoder(smiles)
            tokens = list(selfies.split_selfies(selfies_str))
            return tokens
        except Exception:
            return []

    def __len__(self):
        return len(self.encoded_data)

    def __getitem__(self, idx):
        return torch.tensor(self.encoded_data[idx], dtype=torch.float32)

In [ ]:
# Extract SMILES list and create the dataset and dataloader
# ✅ NEW: Use the 'flp_smiles' column (includes both intra & inter)
smiles_list = flp_all_df['flp_smiles'].dropna().tolist()

dataset = MolecularSELFIESDataset(smiles_list)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
data_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


print(f"✅ Dataset successfully loaded with {len(dataset)} molecules for training.")
print(f"Vocabulary size: {dataset.vocab_size}, Sequence length: {dataset.max_seq_len}")